# DuckDB Explorer: Fundamentals + Series Stores

This notebook validates the current storage model:
- `data/fundamentals/parquet` for endpoint snapshots + analytics datasets
- `data/prices/ibkr_mf_performance_chart` for `price_chart` bars
- `data/sentiment/ibkr_sma_search` for `sentiment_search` bars


In [ ]:
from pathlib import Path
import json
import sqlite3
import duckdb
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 300)
pd.set_option('display.width', 260)

ROOT = Path('/Users/alex/Documents/pystocks')
DUCKDB_PATH = ROOT / 'data' / 'fundamentals' / 'fundamentals.duckdb'
EVENTS_DB_PATH = ROOT / 'data' / 'fundamentals' / 'events.db'
FUND_PARQUET_ROOT = ROOT / 'data' / 'fundamentals' / 'parquet'
PRICE_PARQUET_ROOT = ROOT / 'data' / 'prices' / 'ibkr_mf_performance_chart'
SENTIMENT_PARQUET_ROOT = ROOT / 'data' / 'sentiment' / 'ibkr_sma_search'
TELEMETRY_PATH = ROOT / 'data' / 'research' / 'fundamentals_run_telemetry_latest.json'

assert DUCKDB_PATH.exists(), f'Missing DuckDB file: {DUCKDB_PATH}'
assert EVENTS_DB_PATH.exists(), f'Missing events DB: {EVENTS_DB_PATH}'

con = duckdb.connect(str(DUCKDB_PATH))
print('Connected DuckDB:', DUCKDB_PATH)
print('Fund parquet root exists:', FUND_PARQUET_ROOT.exists())
print('Price parquet root exists:', PRICE_PARQUET_ROOT.exists())
print('Sentiment parquet root exists:', SENTIMENT_PARQUET_ROOT.exists())


def has_relation(name: str) -> bool:
    row = con.execute(
        "SELECT 1 FROM information_schema.tables WHERE lower(table_name)=lower(?) LIMIT 1",
        [name],
    ).fetchone()
    return row is not None


def has_column(relation: str, column: str) -> bool:
    row = con.execute(
        "SELECT 1 FROM information_schema.columns WHERE lower(table_name)=lower(?) AND lower(column_name)=lower(?) LIMIT 1",
        [relation, column],
    ).fetchone()
    return row is not None


def safe_df(query: str, requires=None) -> pd.DataFrame:
    requires = requires or []
    missing = [name for name in requires if not has_relation(name)]
    if missing:
        return pd.DataFrame({'missing_relation': missing})
    return con.sql(query).df()


def count_files(root: Path, pattern: str) -> int:
    if not root.exists():
        return 0
    return len(list(root.glob(pattern)))


def sqlite_df(query: str, params=()):
    with sqlite3.connect(EVENTS_DB_PATH) as conn:
        return pd.read_sql_query(query, conn, params=params)


In [ ]:
safe_df("""
SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_schema='main'
ORDER BY table_name
""")

In [ ]:
print('Endpoint catalog')
display(safe_df("SELECT * FROM endpoint_catalog ORDER BY endpoint", requires=['endpoint_catalog']))

print('Analytics catalog')
display(safe_df("SELECT * FROM analytics_catalog ORDER BY analytics_name", requires=['analytics_catalog']))

print('Price chart series catalog (first 50 conids)')
display(safe_df("SELECT * FROM price_chart_series_catalog ORDER BY conid LIMIT 50", requires=['price_chart_series_catalog']))

print('Sentiment search series catalog (first 50 conids)')
type(safe_df("SELECT * FROM sentiment_search_series_catalog ORDER BY conid LIMIT 50", requires=['sentiment_search_series_catalog']))


In [ ]:
manifest_counts = sqlite_df("""
SELECT endpoint, COUNT(*) AS n_events, MIN(effective_at) AS min_effective_at, MAX(effective_at) AS max_effective_at
FROM endpoint_events
GROUP BY endpoint
ORDER BY endpoint
""")
print('Endpoint events manifest counts (events.db):')
manifest_counts

In [ ]:
if TELEMETRY_PATH.exists():
    tele = json.loads(TELEMETRY_PATH.read_text())
    endpoint_summary = pd.DataFrame(tele.get('endpoint_summary', []))
    endpoint_summary = endpoint_summary.sort_values(['endpoint']).reset_index(drop=True)
    print('Telemetry run stats:')
    display(pd.DataFrame([tele.get('run_stats', {})]))
    print('Telemetry endpoint summary:')
    display(endpoint_summary[['endpoint', 'call_count', 'useful_payload_count', 'useful_payload_rate', 'status_codes']])
else:
    print('Missing telemetry file:', TELEMETRY_PATH)


In [ ]:
inventory = pd.DataFrame([
    {'group': 'fund_endpoints', 'name': 'all', 'files': count_files(FUND_PARQUET_ROOT, 'endpoint=*/year=*/month=*/*.parquet')},
    {'group': 'fund_endpoints', 'name': 'price_chart', 'files': count_files(FUND_PARQUET_ROOT, 'endpoint=price_chart/year=*/month=*/*.parquet')},
    {'group': 'fund_endpoints', 'name': 'sentiment_search', 'files': count_files(FUND_PARQUET_ROOT, 'endpoint=sentiment_search/year=*/month=*/*.parquet')},
    {'group': 'fund_analytics', 'name': 'all', 'files': count_files(FUND_PARQUET_ROOT, 'analytics=*/year=*/month=*/*.parquet')},
    {'group': 'price_series', 'name': 'price_chart', 'files': count_files(PRICE_PARQUET_ROOT, 'conid=*/*.parquet')},
    {'group': 'sentiment_series', 'name': 'sentiment_search', 'files': count_files(SENTIMENT_PARQUET_ROOT, 'conid=*/*.parquet')},
])
print('Parquet inventory')
inventory

In [ ]:
if count_files(FUND_PARQUET_ROOT, 'endpoint=*/year=*/month=*/*.parquet') > 0:
    endpoint_direct = con.sql(f"""
    SELECT endpoint, COUNT(*) AS n_rows
    FROM read_parquet('{(FUND_PARQUET_ROOT / 'endpoint=*/year=*/month=*/*.parquet').as_posix()}', union_by_name=true)
    GROUP BY endpoint
    ORDER BY endpoint
    """).df()
else:
    endpoint_direct = pd.DataFrame(columns=['endpoint', 'n_rows'])

print('Endpoint parquet direct counts (fundamentals/parquet)')
endpoint_direct

In [ ]:
if count_files(FUND_PARQUET_ROOT, 'analytics=*/year=*/month=*/*.parquet') > 0:
    analytics_direct = con.sql(f"""
    SELECT analytics_name, COUNT(*) AS n_rows
    FROM read_parquet('{(FUND_PARQUET_ROOT / 'analytics=*/year=*/month=*/*.parquet').as_posix()}', union_by_name=true)
    GROUP BY analytics_name
    ORDER BY analytics_name
    """).df()
else:
    analytics_direct = pd.DataFrame(columns=['analytics_name', 'n_rows'])

print('Analytics parquet direct counts (fundamentals/parquet)')
analytics_direct

In [ ]:
print('Price chart series sample')
price_sample = safe_df("""
SELECT conid, trade_date, price, open, high, low, close, payload_hash, effective_at
FROM price_chart_series_all
ORDER BY trade_date DESC, conid
LIMIT 200
""", requires=['price_chart_series_all'])

display(price_sample)

print('Price chart endpoint rows present in endpoint_events_all (expected 0 with separate series store):')
safe_df("SELECT COUNT(*) AS n FROM endpoint_events_all WHERE endpoint='price_chart'", requires=['endpoint_events_all'])


In [ ]:
print('Sentiment search series sample')
sent_sample = safe_df("""
SELECT *
FROM sentiment_search_series_all
ORDER BY trade_date DESC, conid
LIMIT 200
""", requires=['sentiment_search_series_all'])
display(sent_sample)

print('Sentiment endpoint rows present in endpoint_events_all (expected 0 with separate series store):')
display(safe_df("SELECT COUNT(*) AS n FROM endpoint_events_all WHERE endpoint='sentiment_search'", requires=['endpoint_events_all']))

forbidden = {'high', 'low', 'price', 'price_change', 'price_change_p', 'open', 'close'}
if has_relation('sentiment_search_series_all'):
    cols = safe_df("""
    SELECT lower(column_name) AS column_name
    FROM information_schema.columns
    WHERE lower(table_name) = 'sentiment_search_series_all'
    ORDER BY column_name
    """, requires=['sentiment_search_series_all'])
    forbidden_present = sorted(set(cols['column_name']).intersection(forbidden)) if 'column_name' in cols else []
    print('Forbidden sentiment columns present (expected []):', forbidden_present)
    display(cols)


In [ ]:
print('Dividends analytics checks')
display(safe_df("""
SELECT analytics_name, COUNT(*) AS n_rows, COUNT(DISTINCT conid) AS n_conids
FROM analytics_rows_all
GROUP BY analytics_name
ORDER BY analytics_name
""", requires=['analytics_rows_all']))

if has_column('analytics_ownership_trade_log', 'action'):
    no_change_check = safe_df("""
    SELECT COUNT(*) AS no_change_rows
    FROM analytics_ownership_trade_log
    WHERE upper(action) = 'NO CHANGE'
    """, requires=['analytics_ownership_trade_log'])
else:
    no_change_check = pd.DataFrame([{'no_change_rows': 0, 'note': 'action column absent (likely no ownership_trade_log rows yet)'}])

lineage_check = safe_df("""
SELECT
  COUNT(*) AS analytics_rows,
  SUM(CASE WHEN e.event_id IS NULL THEN 1 ELSE 0 END) AS missing_event_join,
  SUM(CASE WHEN e.payload_hash IS DISTINCT FROM a.payload_hash THEN 1 ELSE 0 END) AS payload_hash_mismatch
FROM analytics_rows_all a
LEFT JOIN endpoint_events_all e
  ON a.endpoint_event_id = e.event_id
""", requires=['analytics_rows_all', 'endpoint_events_all'])

print('Ownership NO CHANGE rows (expected 0):')
display(no_change_check)
print('Lineage integrity (expected missing_event_join=0, payload_hash_mismatch=0):')
lineage_check


In [ ]:
# Optional cleanup when done
con.close()